<img
  src="https://ai-fullstack-assets.s3.eu-west-3.amazonaws.com/M03-Data_Infrastructure/M03-D01-finpulse-logo.png"
  alt="FinPulse logo"
/>

# FinPulse Data Lake

You joined **FinPulse**, a fintech startup that tracks cryptocurrency markets for retail investors. The company stores daily OHLCV snapshots for ten coins in a JSON file named `crypto_data.json`. For this lab you work from the copy in `src/`. Each coin is a top-level key, and each record has open, high, low, close, and volume fields.

Your manager wants that file off individual laptops. You will push the raw JSON into the team's S3 data lake under `raw/crypto/`, then split each coin into a CSV under `staging/crypto/`. When you finish the tasks below, the data is shared, zoned, and ready for structured warehouse loading later.

You need the `DataEngineers` IAM group, an IAM user with MFA, and access keys in a `.env` file at the **repository root** (same folder as `requirements.txt`), matching the IAM setup you completed in the console. You also need an S3 bucket with `raw/`, `staging/`, and `curated/` prefixes ready in the console.

If you tried `upload_file` with the key `raw/crypto/crypto_data.json` in a short Boto3 demo, this notebook is the full FinPulse version on your own bucket.

<Note type="important">

This notebook uses long-lived access keys in environment variables, which fits local development. In production, prefer an IAM role attached to the workload (for example EC2 or Lambda) so nothing long-lived sits on disk.

</Note>

## Task 1: Connect to S3

Install `boto3` (version 1.28 or newer) and `python-dotenv`. Put a `.env` file at the **repository root** (the same folder as `requirements.txt`) with your IAM keys, then run the setup cell so those values load into the environment. Create a `boto3.Session`, and list your S3 buckets. If the list comes back empty, create a bucket in the console first, then re-run the cell.

<Note type="hint">

The variable names are `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, and optionally `AWS_DEFAULT_REGION`. The code cell walks up from the current working directory and loads the first `.env` it finds, so a root-level file still works when the notebook lives deep in the repo tree. You can still use `export AWS_ACCESS_KEY_ID=...` in the terminal before you launch Jupyter if you prefer not to use a file.

</Note>

<Note type="warning">

Never paste your access key ID or secret access key directly into a notebook cell. If you accidentally commit a key to git, rotate it immediately in the IAM console.

</Note>

In [ ]:
# Run once per environment. In a fresh venv, this pulls a current boto3 line.
!pip install -q "boto3>=1.28" "python-dotenv>=1.0"

Note: you may see a pip resolver warning if another package pins an older botocore. For this course, keep boto3 at 1.28 or newer. If conflicts appear, use a dedicated venv for AWS labs.


In [11]:
import os
import json
import warnings
from pathlib import Path

import boto3
import pandas as pd
from botocore.exceptions import ClientError
from dotenv import load_dotenv

warnings.filterwarnings("ignore")

for _dir in (Path.cwd(), *Path.cwd().parents):
    _env_path = _dir / ".env"
    if _env_path.is_file():
        load_dotenv(_env_path)
        break
else:
    load_dotenv()

if not os.environ.get("AWS_ACCESS_KEY_ID") or not os.environ.get("AWS_SECRET_ACCESS_KEY"):
    raise RuntimeError(
        "Missing AWS_ACCESS_KEY_ID or AWS_SECRET_ACCESS_KEY. "
        "Add a .env file at the repo root, or export both variables in your shell, then re-run this cell."
    )

session = boto3.Session(
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
    region_name=os.environ.get("AWS_DEFAULT_REGION", "eu-north-1"),
)

client = session.client("s3")

response = client.list_buckets()
[b["Name"] for b in response["Buckets"]]

['finpulse-s3-bucket-056793557856-eu-north-1-an']

## Task 2: Land raw data

Point `BUCKET_NAME` at the bucket you created in the S3 console. Upload `src/crypto_data.json` to the raw zone, then list objects under `raw/crypto/` so you can see the key and size.

<Note type="hint">

The third argument to `upload_file` is the S3 key, the full path the object will have inside the bucket. To land the file in the raw zone, the key should start with `raw/`. For this dataset, use `raw/crypto/crypto_data.json`. The `Prefix` argument in `list_objects_v2` filters results to keys that start with that string.

</Note>

In [14]:
BUCKET_NAME = "finpulse-s3-bucket-056793557856-eu-north-1-an"  # Replace with your bucket name

client.upload_file(
    Filename="src/crypto_data.json",
    Bucket=BUCKET_NAME,
    Key="raw/crypto/crypto_data.json",  # raw zone key: raw/crypto/crypto_data.json
)

response = client.list_objects_v2(Bucket=BUCKET_NAME, Prefix="raw/crypto/")
for obj in response.get("Contents", []):
    print(obj["Key"], "-", obj["Size"], "bytes")

raw/crypto/crypto_data.json - 3626805 bytes


## Task 3: Build the staging layer

Load the JSON you already have on disk, build one DataFrame per coin with a `coin` column, stack them, then push one CSV per coin into `staging/crypto/` with `put_object`.

<Note type="hint">

`pd.concat` takes a list of DataFrames and stacks them vertically. Pass `coin_dfs` as the argument and set `ignore_index=True` to get a clean 0-based index. For the S3 key, use an f-string such as `f"staging/crypto/{coin}.csv"`. Unlike `upload_file`, `put_object` accepts an in-memory string or bytes as the `Body`. Calling `df.to_csv(index=False)` returns a string, which is what you need here.

</Note>

<Note type="note">

`put_object` is the right method when your data lives in memory (a string or bytes object). `upload_file` is the right method when your data is already a file on disk. Both end up as S3 objects. The difference is where the bytes come from.

</Note>

In [17]:
with open("src/crypto_data.json") as f:
    crypto = json.load(f)

coin_dfs = []
for coin, records in crypto.items():
    df = pd.DataFrame(records)
    df["coin"] = coin
    coin_dfs.append(df)

# TODO: stack every DataFrame in coin_dfs into one table with pd.concat and ignore_index=True
all_coins = pd.concat(coin_dfs, ignore_index=True)  # stack all DataFrames into one table
print(f"{len(all_coins):,} rows, coins: {all_coins['coin'].nunique()}")

for coin, df in zip(crypto.keys(), coin_dfs):
    client.put_object(
        Bucket=BUCKET_NAME,
        Key=f"staging/crypto/{coin}.csv",
        Body=df.to_csv(index=False),
    )
    print(f"Uploaded staging/crypto/{coin}.csv")

18,017 rows, coins: 10
Uploaded staging/crypto/btc-bitcoin.csv
Uploaded staging/crypto/eth-ethereum.csv
Uploaded staging/crypto/usdt-tether.csv
Uploaded staging/crypto/bnb-binance-coin.csv
Uploaded staging/crypto/hex-hex.csv
Uploaded staging/crypto/usdc-usd-coin.csv
Uploaded staging/crypto/ada-cardano.csv
Uploaded staging/crypto/sol-solana.csv
Uploaded staging/crypto/xrp-xrp.csv
Uploaded staging/crypto/luna-terra.csv


## Task 4: Verify and handle errors

Download the raw object back to a local file, load it with `json.load`, and print the coin keys so you can see the round-trip. Then trigger a `ClientError` on purpose with a key that does not exist, and print the error code from the response.

<Note type="hint">

For the bad key, any string that is not a real object key works. Try `"raw/does_not_exist.json"`. When boto3 raises `ClientError`, the response dict has an `"Error"` key that itself contains a `"Code"` key. S3 returns `NoSuchKey` when using `get_object` on a missing key. With `download_file`, boto3 uses `head_object` internally and returns `404` instead. Either error code means the key does not exist.

</Note>

In [20]:
client.download_file(
    Bucket=BUCKET_NAME,
    Key="raw/crypto/crypto_data.json",
    Filename="crypto_data_downloaded.json",
)

with open("crypto_data_downloaded.json") as f:
    downloaded = json.load(f)

print("Coins in downloaded file:", list(downloaded.keys()))

try:
    client.download_file(
        Bucket=BUCKET_NAME,
        Key="macléquimarchepas",  # use a key that does not exist in your bucket
        Filename="should_not_exist.json",
    )
except ClientError as e:
    error_code = e.response["Error"]["Code"]  # TODO: field name for the error code
    print(f"Caught error: {error_code}")

Coins in downloaded file: ['btc-bitcoin', 'eth-ethereum', 'usdt-tether', 'bnb-binance-coin', 'hex-hex', 'usdc-usd-coin', 'ada-cardano', 'sol-solana', 'xrp-xrp', 'luna-terra']
Caught error: 404


## Stretch Task 1: Add a lifecycle rule

Add a lifecycle rule named `raw-to-ia-30d` that transitions objects under the `raw/` prefix to `STANDARD_IA` after 30 days. Confirm with `get_bucket_lifecycle_configuration`.

<Note type="hint">

A lifecycle rule dict uses this shape: `ID`, `Filter` with `Prefix`, `Status`, and `Transitions`. Wrap that dict as one element of the `Rules` list inside `LifecycleConfiguration` when you call `put_bucket_lifecycle_configuration`.

</Note>

In [21]:
# TODO: use client.put_bucket_lifecycle_configuration to add a lifecycle rule
# that transitions raw/ objects to STANDARD_IA after 30 days.
# The rule ID should be "raw-to-ia-30d".
# Lifecycle rule dict: ID, Filter with Prefix, Status, Transitions list.
# After you apply the rule, call get_bucket_lifecycle_configuration and assign it to lifecycle_check.
lifecycle_rule = {
    "ID": "raw-to-ia-30d",
    "Filter": {"Prefix": "raw/"},
    "Status": "Enabled",
    "Transitions": [{"Days": 30, "StorageClass": "STANDARD_IA"}],
}
client.put_bucket_lifecycle_configuration(
    Bucket=BUCKET_NAME,
    LifecycleConfiguration={"Rules": [lifecycle_rule]},
)
lifecycle_check = client.get_bucket_lifecycle_configuration(Bucket=BUCKET_NAME)
print(lifecycle_check)

{'ResponseMetadata': {'RequestId': 'C3Z7TN3GB2J18KGC', 'HostId': 'gJ0t3fSz44iTTXTGFmGsRS+woUCFDgIgJ/9MV+kZb6rI96eslfmV/Ct3u6Umr/r4vxW+24LsgGpfuBzeE39DbmMb69E01asR', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amz-id-2': 'gJ0t3fSz44iTTXTGFmGsRS+woUCFDgIgJ/9MV+kZb6rI96eslfmV/Ct3u6Umr/r4vxW+24LsgGpfuBzeE39DbmMb69E01asR', 'x-amz-request-id': 'C3Z7TN3GB2J18KGC', 'date': 'Wed, 23 Sep 2026 15:18:09 GMT', 'x-amz-transition-default-minimum-object-size': 'all_storage_classes_128K', 'content-length': '313', 'server': 'AmazonS3'}, 'RetryAttempts': 0}, 'TransitionDefaultMinimumObjectSize': 'all_storage_classes_128K', 'Rules': [{'ID': 'raw-to-ia-30d', 'Filter': {'Prefix': 'raw/'}, 'Status': 'Enabled', 'Transitions': [{'Days': 30, 'StorageClass': 'STANDARD_IA'}]}]}


## Stretch Task 2: Curated summary

From `all_coins`, compute per-coin average and maximum close price, then upload `curated/crypto/summary.csv`.

<Note type="hint">

`all_coins.groupby("coin")["close"].agg(["mean", "max"])` gives you a DataFrame indexed by coin. Call `.reset_index()` to turn the index back into a column, then rename with `.rename(columns={"mean": "avg_close", "max": "max_close"})` to match the column names below. For the upload, the S3 key is `curated/crypto/summary.csv`.

</Note>

In [26]:
# TODO: using all_coins, compute per-coin average and max close price.
# The result should be a DataFrame with columns: coin, avg_close, max_close.
# TODO: upload the summary DataFrame as a CSV to curated/crypto/summary.csv
# using client.put_object (same pattern as Task 3).
clotures_moyen = all_coins.groupby('coin')['close'].agg(["mean", "max"]).reset_index()
clotures_moyen.columns = ["coin", "avg_close", "max_close"]
clotures_moyen[["avg_close", "max_close"]] = clotures_moyen[["avg_close", "max_close"]]
print(clotures_moyen)



               coin    avg_close     max_close
0       ada-cardano     0.443466      2.968622
1  bnb-binance-coin    97.417116    678.647111
2       btc-bitcoin  7912.605595  67589.768671
3      eth-ethereum   622.884154   4821.848354
4           hex-hex     0.068430      0.487742
5        luna-terra    10.125173     99.884867
6        sol-solana    60.087259    258.881644
7     usdc-usd-coin     1.003053      1.041809
8       usdt-tether     1.001417      1.211100
9           xrp-xrp     0.278041      3.377810
